# AutoGen: Building Conversational Multi-Agent Systems

This notebook is a comprehensive, practical guide to **AutoGen** (Microsoft's framework for building LLM-powered multi-agent conversation systems). AutoGen agents communicate through structured conversations, making it well-suited for iterative, back-and-forth problem solving.

All examples use **GPT-4o-mini** as the language model and **text-embedding-3-small** for embedding operations.

---

## Table of Contents

1. Installation and Environment Setup
2. Core Concepts: How AutoGen Differs from CrewAI
3. Agent Types and Their Properties
4. ConversableAgent Properties Deep Dive
5. Conversation Patterns
6. Implementation 1 — Two-Agent Code Generation and Review
7. Implementation 2 — GroupChat: Multi-Agent Product Planning
8. Implementation 3 — Math Tutoring with Code Execution
9. Implementation 4 — Automated Data Analysis Pipeline
11. Advanced: Tool / Function Calling
12. Advanced: Nested Chats
13. Advanced: Human Proxy with Termination Conditions
14. Advanced: Custom Reply Functions
15. Best Practices and Common Pitfalls

---
## 1. Installation and Environment Setup

In [ ]:
# Install AutoGen and supporting libraries
%pip install pyautogen openai python-dotenv chromadb sentence-transformers --quiet

# For code execution sandbox support
%pip install docker --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 890.9 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 53.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.3/119.3 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.9/101.9 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 75.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 63.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6

In [ ]:
# Re-install AutoGen to ensure all dependencies are met
%pip install pyautogen --quiet

In [ ]:
from google.colab import userdata

# Fetch API key securely from Colab Secrets
api_key = userdata.get("OPENAI_API_KEY")

assert api_key, "OPENAI_API_KEY must be set in Colab Secrets."

# AutoGen config_list
config_list = [
    {
        "model": "gpt-4o-mini",
        "api_key": api_key,
    }
]

# LLM configuration
llm_config = {
    "config_list": config_list,
    "temperature": 0.2,
    "cache_seed": None,  # Set to an int like 42 for reproducibility
}

print("AutoGen environment configured.")
print(f"Model: {config_list[0]['model']}")

AutoGen environment configured.
Model: gpt-4o-mini


---
## 2. Core Concepts: How AutoGen Differs from CrewAI

| Aspect | CrewAI | AutoGen |
|---|---|---|
| **Paradigm** | Role-based task assignment | Conversational message passing |
| **Execution** | Tasks run sequentially or hierarchically | Agents converse in turns until termination |
| **Human interaction** | Optional pause on specific tasks | Central via `HumanProxyAgent` |
| **Code execution** | External tools only | Native sandboxed code execution |
| **Multi-agent** | Crew orchestrates agents | GroupChat with a manager routes messages |
| **Best for** | Structured pipelines with defined outputs | Iterative problem solving requiring dialogue |

### The AutoGen Conversation Model

In AutoGen, agents communicate by sending and receiving **messages** (strings or dicts). The conversation continues until a **termination condition** is met — typically when an agent sends a message containing a specific string (e.g., `TERMINATE`) or a maximum number of turns is reached.


---
## 3. Agent Types and Their Properties

AutoGen ships with several agent classes, each suited to different roles:

| Agent Class | Purpose |
|---|---|
| `ConversableAgent` | The base class. All agents inherit from this. |
| `AssistantAgent` | An LLM-backed agent optimized for AI assistance tasks. |
| `UserProxyAgent` | Simulates a human user. Can execute code locally or in Docker. |
| `GroupChatManager` | Orchestrates which agent speaks next in a group conversation. |
| `RetrieveAssistantAgent` | AssistantAgent extended with RAG capabilities. |
| `RetrieveUserProxyAgent` | UserProxyAgent extended with document retrieval. |

In [ ]:
!pip install autogen-ext[openai]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.4/331.4 kB 2.4 MB/s eta 0:00:00


---
## 4. ConversableAgent Properties Deep Dive

In [ ]:
from google.colab import userdata
from autogen_agentchat.agents import AssistantAgent
from autogen_ext.models.openai import OpenAIChatCompletionClient

# Get API key from Colab secrets
api_key = userdata.get("OPENAI_API_KEY")
assert api_key, "Missing OPENAI_API_KEY in Colab Secrets"

# Create model client (NEW way)
model_client = OpenAIChatCompletionClient(
    model="gpt-4o-mini",
    api_key=api_key,
)

# Create agent (UPDATED)
fully_configured = AssistantAgent(
    name="DataScienceExpert",
    system_message=(
        "You are a senior data scientist with expertise in Python, pandas, "
        "scikit-learn, and statistical modeling. "
        "Explain your approach clearly and end your response with TERMINATE."
    ),
    model_client=model_client,
)

print("✅ Agent configured:", fully_configured.name)

✅ Agent configured: DataScienceExpert


### ConversableAgent Property Summary

| Property | Type | Description |
|---|---|---|
| `name` | str | Unique agent identifier |
| `system_message` | str | Persona and behavior instructions |
| `llm_config` | dict / False | LLM configuration or disabled |
| `code_execution_config` | dict / False | Enable local or Docker code execution |
| `human_input_mode` | str | `NEVER`, `ALWAYS`, or `TERMINATE` |
| `is_termination_msg` | callable | Function to detect conversation end |
| `max_consecutive_auto_reply` | int | Cap on auto replies before requesting human |
| `function_map` | dict | Maps tool names to Python callables |
| `default_auto_reply` | str | Message sent when the agent has nothing to say |

---
## 5. Conversation Patterns

AutoGen supports several conversation topologies:

In [ ]:
# Pattern overview (not executed — for reference)

patterns = {
    "Two-Agent Chat": {
        "description": "The simplest pattern. One agent initiates, the other responds. "
                       "Back-and-forth until termination.",
        "method": "agent_a.initiate_chat(agent_b, message='...')",
    },
    "Sequential Chat": {
        "description": "A series of two-agent chats where the summary of one "
                       "is passed as context to the next.",
        "method": "agent.initiate_chats([chat_1_config, chat_2_config, ...])",
    },
    "Group Chat": {
        "description": "Multiple agents in one conversation. A GroupChatManager "
                       "decides which agent speaks next using a selection strategy.",
        "method": "GroupChat + GroupChatManager",
    },
    "Nested Chat": {
        "description": "An agent spawns a private sub-conversation with other agents "
                       "and returns the result to the parent conversation.",
        "method": "agent.register_nested_chats([nested_config])",
    },
}

for name, info in patterns.items():
    print(f"Pattern: {name}")
    print(f"  Description : {info['description']}")
    print(f"  Method      : {info['method']}")
    print()

Pattern: Two-Agent Chat
  Description : The simplest pattern. One agent initiates, the other responds. Back-and-forth until termination.
  Method      : agent_a.initiate_chat(agent_b, message='...')

Pattern: Sequential Chat
  Description : A series of two-agent chats where the summary of one is passed as context to the next.
  Method      : agent.initiate_chats([chat_1_config, chat_2_config, ...])

Pattern: Group Chat
  Description : Multiple agents in one conversation. A GroupChatManager decides which agent speaks next using a selection strategy.
  Method      : GroupChat + GroupChatManager

Pattern: Nested Chat
  Description : An agent spawns a private sub-conversation with other agents and returns the result to the parent conversation.
  Method      : agent.register_nested_chats([nested_config])



---
## 6. Implementation 1 — Two-Agent Code Generation and Review

**Scenario:** A software engineer agent writes Python code and a code reviewer agent audits it for bugs, style, and correctness. They iterate until the reviewer approves.

In [ ]:
# ================================
# INSTALL (run once in Colab)
# ================================
!pip install pyautogen autogen-ext[openai] --upgrade


# ================================
# IMPORTS
# ================================
import asyncio
from google.colab import userdata

from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_ext.models.openai import OpenAIChatCompletionClient


# ================================
# SETUP MODEL CLIENT
# ================================
api_key = userdata.get("OPENAI_API_KEY")
assert api_key, "Missing OPENAI_API_KEY in Colab Secrets"

model_client = OpenAIChatCompletionClient(
    model="gpt-4o-mini",
    api_key=api_key,
)


# ================================
# AGENTS
# ================================

# --- Engineer Agent ---
engineer = AssistantAgent(
    name="SoftwareEngineer",
    system_message=(
        "You are a senior Python engineer. "
        "Write complete, production-ready Python code with type hints, "
        "docstrings, and proper error handling. "
        "Revise code based on reviewer feedback. "
        "When reviewer says APPROVED, reply TERMINATE."
    ),
    model_client=model_client,
)

# --- Reviewer Agent ---
reviewer = AssistantAgent(
    name="CodeReviewer",
    system_message=(
        "You are a principal engineer reviewing Python code. "
        "Check correctness, type hints, docstrings, validation, and PEP8. "
        "If issues exist, clearly list them and request fixes. "
        "If correct, respond EXACTLY with: APPROVED."
    ),
    model_client=model_client,
)


# ================================
# TEAM (replaces initiate_chat)
# ================================
team = RoundRobinGroupChat(
    participants=[reviewer, engineer],
    max_turns=6,
)


# ================================
# TASK
# ================================
task = (
    "Implement a Python class `BankAccount` with:\n"
    "1. Constructor: owner (str), initial_balance (float=0.0)\n"
    "2. Methods: deposit, withdraw, get_balance, transaction_history\n"
    "3. deposit/withdraw must validate amount > 0\n"
    "4. withdraw raises InsufficientFundsError if balance < 0\n"
    "5. All transactions stored with timestamps"
)


# ================================
# RUN (Colab / Jupyter FIX)
# ================================
result = await team.run(task=task)

print("\n✅ Conversation complete\n")

for msg in result.messages:
    print(f"{msg.source}:")
    print(msg.content)
    print("-" * 50)


✅ Conversation complete

user:
Implement a Python class `BankAccount` with:
1. Constructor: owner (str), initial_balance (float=0.0)
2. Methods: deposit, withdraw, get_balance, transaction_history
3. deposit/withdraw must validate amount > 0
4. withdraw raises InsufficientFundsError if balance < 0
5. All transactions stored with timestamps
--------------------------------------------------
CodeReviewer:
Here is a `BankAccount` class implementation based on your requirements:

```python
from datetime import datetime
from typing import List, Tuple

class InsufficientFundsError(Exception):
    """Custom exception raised when an account does not have enough funds for withdrawal."""
    pass

class BankAccount:
    def __init__(self, owner: str, initial_balance: float = 0.0) -> None:
        """Initialize a new bank account with the owner's name and an initial balance.
        
        Args:
            owner (str): The name of the account owner.
            initial_balance (float, optiona

---
## 7. Implementation 2 — GroupChat: Multi-Agent Product Planning

**Scenario:** A product launch is being planned. A Product Manager, UX Designer, Backend Engineer, and Marketing Lead debate requirements in a group chat. A manager routes the conversation.

In [ ]:
# ================================
# INSTALL (run once)
# ================================
# !pip install pyautogen autogen-ext[openai] --upgrade


# ================================
# IMPORTS
# ================================
from google.colab import userdata
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_ext.models.openai import OpenAIChatCompletionClient


# ================================
# MODEL CLIENT
# ================================
api_key = userdata.get("OPENAI_API_KEY")

model_client = OpenAIChatCompletionClient(
    model="gpt-4o-mini",
    api_key=api_key,
)


# ================================
# AGENTS (SPECIALISTS)
# ================================

product_manager = AssistantAgent(
    name="ProductManager",
    system_message=(
        "You are a product manager. Define user stories, acceptance criteria, "
        "and prioritize features. Focus on user value and business impact."
    ),
    model_client=model_client,
)

ux_designer = AssistantAgent(
    name="UXDesigner",
    system_message=(
        "You are a UX designer. Focus on usability, accessibility, and interface design. "
        "Propose clear UX decisions."
    ),
    model_client=model_client,
)

backend_engineer = AssistantAgent(
    name="BackendEngineer",
    system_message=(
        "You are a backend engineer. Evaluate technical feasibility, APIs, scalability, "
        "security, and timelines."
    ),
    model_client=model_client,
)

marketing_lead = AssistantAgent(
    name="MarketingLead",
    system_message=(
        "You are a marketing lead. Focus on go-to-market strategy, messaging, "
        "target audience, and differentiation."
    ),
    model_client=model_client,
)

# Optional: Stakeholder agent (replaces UserProxyAgent)
stakeholder = AssistantAgent(
    name="Stakeholder",
    system_message=(
        "You are the business stakeholder guiding the discussion. "
        "Ensure the team reaches a clear consensus and ends with MEETING ADJOURNED."
    ),
    model_client=model_client,
)


# ================================
# TEAM (REPLACES GROUPCHAT)
# ================================
team = RoundRobinGroupChat(
    participants=[
        stakeholder,
        product_manager,
        ux_designer,
        backend_engineer,
        marketing_lead,
    ],
    max_turns=12,
)


# ================================
# TASK (MEETING AGENDA)
# ================================
task = (
    "We are planning the launch of a B2B SaaS expense management tool "
    "for SMBs (10–200 employees), launching in 90 days.\n\n"
    "Discuss:\n"
    "1. Three must-have MVP features\n"
    "2. Key technical architecture decisions\n"
    "3. Primary go-to-market channel\n"
    "4. Biggest risk to timeline\n\n"
    "Conclude with a consensus summary.\n"
    "End with MEETING ADJOURNED."
)


# ================================
# RUN (Colab-safe)
# ================================
result = await team.run(task=task)

print("\n✅ Meeting complete\n")

for msg in result.messages:
    print(f"{msg.source}:")
    print(msg.content)
    print("-" * 50)


✅ Meeting complete

user:
We are planning the launch of a B2B SaaS expense management tool for SMBs (10–200 employees), launching in 90 days.

Discuss:
1. Three must-have MVP features
2. Key technical architecture decisions
3. Primary go-to-market channel
4. Biggest risk to timeline

Conclude with a consensus summary.
End with MEETING ADJOURNED.
--------------------------------------------------
Stakeholder:
Thank you for organizing this discussion. Let's take a systematic approach to tackle each point regarding our upcoming B2B SaaS expense management tool for SMBs.

### 1. Three Must-Have MVP Features:
- **Expense Tracking and Reporting**: Users should be able to capture expenses by uploading receipts and categorizing them for easy reporting and analysis. This feature serves as the core functionality and ensures users can manage expenses efficiently.
  
- **Integration with Popular Accounting Software**: To increase usability and improve customer adoption, seamless integration with 

---
## 8. Implementation 3 — Math Tutoring with Code Execution

**Scenario:** A student agent poses a statistics problem. An AI tutor explains the approach and writes Python code. The UserProxyAgent executes the code and returns the result.

In [ ]:
!pip install -U "autogen-ext[docker]"
from autogen_ext.code_executors.local import LocalCommandLineCodeExecutor

In [ ]:
!pip install asyncio-atexit

In [ ]:
# ================================
# INSTALL (run once)
# ================================
!pip install pyautogen autogen-ext[openai] autogen-ext[code] --upgrade


# ================================
# IMPORTS
# ================================
import os
from google.colab import userdata

from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_ext.models.openai import OpenAIChatCompletionClient

# 🔥 NEW IMPORTS
from autogen_ext.code_executors.local import LocalCommandLineCodeExecutor
from autogen_ext.tools.code_execution import PythonCodeExecutionTool


# ================================
# WORKSPACE
# ================================
os.makedirs("./autogen_workspace", exist_ok=True)


# ================================
# MODEL CLIENT
# ================================
api_key = userdata.get("OPENAI_API_KEY")

model_client = OpenAIChatCompletionClient(
    model="gpt-4o-mini",
    api_key=api_key,
)


# ================================
# EXECUTOR (NEW REQUIRED LAYER)
# ================================
executor_backend = LocalCommandLineCodeExecutor(
    work_dir="./autogen_workspace",
    timeout=30,
)


# ================================
# TOOL (WRAPS EXECUTOR)
# ================================
code_tool = PythonCodeExecutionTool(
    executor=executor_backend
)


# ================================
# AGENTS
# ================================
tutor = AssistantAgent(
    name="StatisticsTutor",
    system_message=(
        "You are an expert statistics professor.\n"
        "1. Explain concept\n"
        "2. Write Python code (numpy, scipy, matplotlib)\n"
        "3. Interpret results\n"
        "Wrap code in ONE python block.\n"
        "End with TERMINATE."
    ),
    model_client=model_client,
)

executor_agent = AssistantAgent(
    name="CodeExecutor",
    system_message="Execute Python code and return ONLY output.",
    model_client=model_client,
    tools=[code_tool],
)


# ================================
# TEAM
# ================================
team = RoundRobinGroupChat(
    participants=[tutor, executor_agent],
    max_turns=8,
)


# ================================
# TASK
# ================================
task = (
    "Section A: [72, 85, 90, 68, 77, 95, 82, 78, 88, 74]\n"
    "Section B: [65, 70, 88, 92, 76, 81, 69, 83, 79, 95]\n\n"
    "1. Statistical significance test (alpha=0.05)\n"
    "2. Cohen's d\n"
    "3. Boxplot"
)


# ================================
# RUN (Colab-safe)
# ================================
result = await team.run(task=task)

print("\n✅ Done\n")

for msg in result.messages:
    print(f"{msg.source}:")
    print(msg.content)
    print("-" * 50)

/tmp/ipykernel_1473/4018394800.py:42: UserWarning: Using LocalCommandLineCodeExecutor may execute code on the local machine which can be unsafe. For security, it is recommended to use DockerCommandLineCodeExecutor instead. To install Docker, visit: https://docs.docker.com/get-docker/
  executor_backend = LocalCommandLineCodeExecutor(



✅ Done

user:
Section A: [72, 85, 90, 68, 77, 95, 82, 78, 88, 74]
Section B: [65, 70, 88, 92, 76, 81, 69, 83, 79, 95]

1. Statistical significance test (alpha=0.05)
2. Cohen's d
3. Boxplot
--------------------------------------------------
StatisticsTutor:
1. **Concept Explanation:**
   In order to compare the means of two independent groups (Section A and Section B in this case), we typically use a t-test. The null hypothesis (H0) states that there is no significant difference between the means of the two sections, while the alternative hypothesis (H1) states that there is a significant difference. We will use an alpha level of 0.05 to determine if we should reject the null hypothesis.

   Cohen's d is a measure of effect size, which indicates the standardized difference between two means. It is calculated as the difference between the means divided by the pooled standard deviation. A larger Cohen's d value indicates a larger effect size.

   Finally, we can visualize the data using 

---
## 9. Implementation 4 — Automated Data Analysis Pipeline

**Scenario:** A data analyst agent writes code to analyze a sales dataset, produces statistics and a chart, and a reporter agent writes an executive summary of the findings.

In [ ]:
import pandas as pd
import os

# Create workspace
os.makedirs("./autogen_workspace", exist_ok=True)

# Create dataset
sales_data = pd.DataFrame({
    "month": ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
              "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"],
    "revenue": [420000, 385000, 510000, 490000, 610000, 580000,
                540000, 620000, 700000, 680000, 810000, 950000],
    "units_sold": [1200, 1050, 1480, 1390, 1720, 1680,
                   1540, 1790, 2010, 1950, 2340, 2680],
    "region": ["North", "North", "South", "South", "East", "East",
               "West", "West", "North", "South", "East", "West"],
    "customer_satisfaction": [4.2, 4.0, 4.5, 4.3, 4.6, 4.4,
                             4.1, 4.7, 4.8, 4.5, 4.9, 4.8],
})

# Save CSV
file_path = "./autogen_workspace/sales_2023.csv"
sales_data.to_csv(file_path, index=False)

print(f"✅ Sample dataset created at {file_path}\n")
print(sales_data.head())

✅ Sample dataset created at ./autogen_workspace/sales_2023.csv

  month  revenue  units_sold region  customer_satisfaction
0   Jan   420000        1200  North                    4.2
1   Feb   385000        1050  North                    4.0
2   Mar   510000        1480  South                    4.5
3   Apr   490000        1390  South                    4.3
4   May   610000        1720   East                    4.6


In [ ]:
# ================================
# INSTALL (run once)
# ================================
!pip install pyautogen autogen-ext[openai] autogen-ext[code] asyncio-atexit --upgrade


# ================================
# IMPORTS
# ================================
import os
from google.colab import userdata

from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_ext.code_executors.local import LocalCommandLineCodeExecutor
from autogen_ext.tools.code_execution import PythonCodeExecutionTool


# ================================
# MODEL CLIENT
# ================================
api_key = userdata.get("OPENAI_API_KEY")

model_client = OpenAIChatCompletionClient(
    model="gpt-4o-mini",
    api_key=api_key,
)


# ================================
# EXECUTION TOOL
# ================================
executor_backend = LocalCommandLineCodeExecutor(
    work_dir="./autogen_workspace",
    timeout=60,
)

code_tool = PythonCodeExecutionTool(
    executor=executor_backend
)


# ================================
# AGENTS
# ================================

# --- Data Analyst ---
data_analyst = AssistantAgent(
    name="DataAnalyst",
    system_message=(
        "You are a senior data analyst.\n"
        "1. Load CSV\n"
        "2. Compute requested metrics\n"
        "3. Provide 3 key insights\n"
        "4. Generate chart and save PNG\n"
        "Use pandas, matplotlib, seaborn.\n"
        "Wrap code in ONE python block.\n"
        "End with ANALYSIS COMPLETE."
    ),
    model_client=model_client,
)

# --- Code Executor ---
executor_agent = AssistantAgent(
    name="CodeExecutor",
    system_message=(
        "Execute Python code from previous message. "
        "Return ONLY output. Do not explain."
    ),
    model_client=model_client,
    tools=[code_tool],
)

# --- Reporter ---
reporter = AssistantAgent(
    name="ExecutiveReporter",
    system_message=(
        "You are a business intelligence writer.\n"
        "Write under 300 words:\n"
        "Overview (2 sentences)\n"
        "Key Findings (3 bullets with numbers)\n"
        "Recommendations (2 actions)\n"
        "End with TERMINATE."
    ),
    model_client=model_client,
)


# ================================
# TEAM (PIPELINE FLOW)
# ================================
team = RoundRobinGroupChat(
    participants=[data_analyst, executor_agent, reporter],
    max_turns=8,
)


# ================================
# TASK
# ================================
task = (
    "Analyze dataset at ./autogen_workspace/sales_2023.csv\n\n"
    "Compute:\n"
    "1. Total revenue and units\n"
    "2. Month-over-month revenue growth\n"
    "3. Revenue by region\n"
    "4. Correlation between satisfaction and revenue\n\n"
    "Save chart as monthly_revenue.png\n\n"
    "Then produce executive summary."
)


# ================================
# RUN (Colab-safe)
# ================================
result = await team.run(task=task)

print("\n✅ Pipeline complete\n")

for msg in result.messages:
    print(f"{msg.source}:")
    print(msg.content)
    print("-" * 50)

/tmp/ipykernel_1473/1067575186.py:34: UserWarning: Using LocalCommandLineCodeExecutor may execute code on the local machine which can be unsafe. For security, it is recommended to use DockerCommandLineCodeExecutor instead. To install Docker, visit: https://docs.docker.com/get-docker/
  executor_backend = LocalCommandLineCodeExecutor(



✅ Pipeline complete

user:
Analyze dataset at ./autogen_workspace/sales_2023.csv

Compute:
1. Total revenue and units
2. Month-over-month revenue growth
3. Revenue by region
4. Correlation between satisfaction and revenue

Save chart as monthly_revenue.png

Then produce executive summary.
--------------------------------------------------
DataAnalyst:
```python
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load the CSV file
file_path = './autogen_workspace/sales_2023.csv'
data = pd.read_csv(file_path)

# 1. Compute Total Revenue and Units
total_revenue = data['Revenue'].sum()
total_units = data['Units Sold'].sum()

# 2. Compute Month-over-Month Revenue Growth
data['Month'] = pd.to_datetime(data['Date']).dt.to_period('M')
monthly_revenue = data.groupby('Month')['Revenue'].sum()
month_over_month_growth = monthly_revenue.pct_change() * 100

# 3. Compute Revenue by Region
revenue_by_region = data.groupby('Region')['Revenue'].sum()

# 4. Compute Correlation b

---
## 10
. Advanced: Tool / Function Calling

AutoGen supports OpenAI-style function calling. You register functions on the agent and they are called when the LLM decides to use them.

In [ ]:
# ================================
# INSTALL (run once)
# ================================
!pip install pyautogen autogen-ext[openai] --upgrade


# ================================
# IMPORTS
# ================================
from datetime import datetime
from google.colab import userdata

from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_ext.models.openai import OpenAIChatCompletionClient


# ================================
# TOOL FUNCTIONS
# ================================
def get_current_stock_price(ticker: str) -> dict:
    mock_prices = {
        "AAPL": {"price": 189.42, "change_pct": 0.82, "volume": "52.3M"},
        "MSFT": {"price": 415.20, "change_pct": -0.31, "volume": "18.9M"},
        "GOOGL": {"price": 178.65, "change_pct": 1.24, "volume": "21.1M"},
        "NVDA": {"price": 875.30, "change_pct": 2.15, "volume": "41.7M"},
    }
    data = mock_prices.get(ticker.upper(), {"error": f"Ticker {ticker} not found"})
    return {"ticker": ticker.upper(), "timestamp": datetime.now().isoformat(), **data}


def calculate_portfolio_value(holdings: list) -> dict:
    mock_prices = {"AAPL": 189.42, "MSFT": 415.20, "GOOGL": 178.65, "NVDA": 875.30}
    total = 0.0
    breakdown = []

    for h in holdings:
        ticker = h["ticker"].upper()
        shares = h["shares"]
        price = mock_prices.get(ticker, 0)
        value = price * shares
        total += value

        breakdown.append({
            "ticker": ticker,
            "shares": shares,
            "price": price,
            "value": round(value, 2)
        })

    return {
        "total_value": round(total, 2),
        "breakdown": breakdown
    }


# ================================
# MODEL CLIENT
# ================================
api_key = userdata.get("OPENAI_API_KEY")

model_client = OpenAIChatCompletionClient(
    model="gpt-4o-mini",
    api_key=api_key,
)


# ================================
# AGENT WITH TOOLS (NEW WAY)
# ================================
portfolio_advisor = AssistantAgent(
    name="PortfolioAdvisor",
    system_message=(
        "You are a portfolio advisor.\n"
        "Use tools to fetch stock prices and calculate portfolio value.\n"
        "Show calculations clearly.\n"
        "End with TERMINATE."
    ),
    model_client=model_client,
    tools=[get_current_stock_price, calculate_portfolio_value],  # ✅ direct functions
)


# ================================
# TEAM (single-agent loop)
# ================================
team = RoundRobinGroupChat(
    participants=[portfolio_advisor],
    max_turns=5,
)


# ================================
# TASK
# ================================
task = (
    "I hold:\n"
    "- 50 shares of AAPL\n"
    "- 30 shares of MSFT\n"
    "- 15 shares of NVDA\n\n"
    "1. Get current price for each\n"
    "2. Calculate total portfolio value\n"
    "3. Identify largest holding percentage"
)


# ================================
# RUN (Colab-safe)
# ================================
result = await team.run(task=task)

print("\n✅ Done\n")

for msg in result.messages:
    print(f"{msg.source}:")
    print(msg.content)
    print("-" * 50)


✅ Done

user:
I hold:
- 50 shares of AAPL
- 30 shares of MSFT
- 15 shares of NVDA

1. Get current price for each
2. Calculate total portfolio value
3. Identify largest holding percentage
--------------------------------------------------
PortfolioAdvisor:
[FunctionCall(id='call_LqxZHv5xM94yMOyOIyaXvJiG', arguments='{"ticker": "AAPL"}', name='get_current_stock_price'), FunctionCall(id='call_8LvSUurax2MLoi7prEk3MYdw', arguments='{"ticker": "MSFT"}', name='get_current_stock_price'), FunctionCall(id='call_DxbjJHVeHSkVzBznvgzlDdxj', arguments='{"ticker": "NVDA"}', name='get_current_stock_price')]
--------------------------------------------------
PortfolioAdvisor:
[FunctionExecutionResult(content="{'ticker': 'AAPL', 'timestamp': '2026-07-22T05:29:40.176811', 'price': 189.42, 'change_pct': 0.82, 'volume': '52.3M'}", name='get_current_stock_price', call_id='call_LqxZHv5xM94yMOyOIyaXvJiG', is_error=False), FunctionExecutionResult(content="{'ticker': 'MSFT', 'timestamp': '2026-07-22T05:29:40.1

---
## 12. Advanced: Nested Chats

Nested chats allow an agent to spawn a private sub-conversation with other agents and return the result to the parent conversation — useful for modular reasoning.

In [ ]:
# ================================
# INSTALL (run once)
# ================================
# !pip install pyautogen autogen-ext[openai] --upgrade


# ================================
# IMPORTS
# ================================
from google.colab import userdata

from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_ext.models.openai import OpenAIChatCompletionClient


# ================================
# MODEL CLIENT
# ================================
api_key = userdata.get("OPENAI_API_KEY")

model_client = OpenAIChatCompletionClient(
    model="gpt-4o-mini",
    api_key=api_key,
)


# ================================
# AGENTS
# ================================

# --- Financial Specialist ---
financial_modeler = AssistantAgent(
    name="FinancialModeler",
    system_message=(
        "You are a financial modeling specialist.\n"
        "Compute:\n"
        "- Break-even units\n"
        "- Gross margin\n"
        "- 12-month revenue projection\n"
        "Return results in a structured table.\n"
        "End with MODEL COMPLETE."
    ),
    model_client=model_client,
)

# --- Business Consultant ---
business_consultant = AssistantAgent(
    name="BusinessConsultant",
    system_message=(
        "You are a business consultant.\n"
        "When solving a pricing problem:\n"
        "1. First consult FinancialModeler\n"
        "2. Use its output\n"
        "3. Provide final recommendation\n"
        "Only return the final business answer (not raw model details).\n"
        "End with TERMINATE."
    ),
    model_client=model_client,
)


# ================================
# TEAM (simulates nested flow)
# ================================
team = RoundRobinGroupChat(
    participants=[financial_modeler, business_consultant],
    max_turns=4,
)


# ================================
# TASK
# ================================
task = (
    "Client request:\n"
    "SaaS pricing scenario:\n"
    "- Cost per customer: $12/month\n"
    "- Price: $49/month\n"
    "- Growth: 10 customers → 200 customers in 12 months\n\n"
    "Provide pricing evaluation and financial outlook."
)


# ================================
# RUN (Colab-safe)
# ================================
result = await team.run(task=task)

print("\n✅ Done\n")

for msg in result.messages:
    print(f"{msg.source}:")
    print(msg.content)
    print("-" * 50)


✅ Done

user:
Client request:
SaaS pricing scenario:
- Cost per customer: $12/month
- Price: $49/month
- Growth: 10 customers → 200 customers in 12 months

Provide pricing evaluation and financial outlook.
--------------------------------------------------
FinancialModeler:
To evaluate the pricing scenario for the SaaS (Software as a Service), let’s compute the following metrics:

1. **Break-even units**: This is the number of customers needed to cover all costs.
2. **Gross margin**: This represents the difference between the revenue and the cost of goods sold (COGS), expressed as a percentage.
3. **12-month revenue projection**: This indicates the expected revenue generated from customers over the next 12 months.

**Given Data:**
- Cost per customer per month: $12
- Price per customer per month: $49
- Growth: From 10 customers to 200 customers over 12 months

### Calculations

1. **Break-even units:**
   - Break-even point (in customers) = Fixed Costs / (Price - Cost per Customer)
  

---
## 13. Advanced: Human Proxy with Termination Conditions

In [ ]:
# ================================
# INSTALL (run once)
# ================================
# !pip install pyautogen autogen-ext[openai] --upgrade


# ================================
# IMPORTS
# ================================
from google.colab import userdata

from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_ext.models.openai import OpenAIChatCompletionClient


# ================================
# MODEL CLIENT
# ================================
api_key = userdata.get("OPENAI_API_KEY")

model_client = OpenAIChatCompletionClient(
    model="gpt-4o-mini",
    api_key=api_key,
)


# ================================
# AGENT
# ================================
ai_drafter = AssistantAgent(
    name="ContractDrafter",
    system_message=(
        "You draft professional business contracts.\n"
        "After completing, end with: DRAFT COMPLETE."
    ),
    model_client=model_client,
)


# ================================
# TEAM (single agent loop)
# ================================
team = RoundRobinGroupChat(
    participants=[ai_drafter],
    max_turns=3,
)


# ================================
# STEP 1: GENERATE DRAFT
# ================================
task = (
    "Draft a one-page NDA between Acme Corp (disclosing party) "
    "and a freelance consultant (receiving party). "
    "Include: confidential info definition, obligations, exclusions, "
    "2-year term, New York law."
)

result = await team.run(task=task)

print("\n📄 AI Draft:\n")

final_msg = result.messages[-1].content
print(final_msg)


# ================================
# STEP 2: HUMAN REVIEW (MANUAL)
# ================================
if "DRAFT COMPLETE" in final_msg:
    print("\n🧑‍⚖️ Review required.")
    feedback = input("Enter feedback (or type 'approve'): ")

    if feedback.lower() != "approve":
        # ================================
        # STEP 3: REVISE BASED ON FEEDBACK
        # ================================
        revision_task = (
            f"Revise the NDA based on this feedback:\n{feedback}"
        )

        revision_result = await team.run(task=revision_task)

        print("\n📄 Revised Draft:\n")
        print(revision_result.messages[-1].content)

    else:
        print("\n✅ Draft approved.")


📄 AI Draft:

**NON-DISCLOSURE AGREEMENT**

This Non-Disclosure Agreement ("Agreement") is made and entered into as of the ___ day of __________, 20___, by and between Acme Corp, having its principal place of business at [Insert Address] ("Disclosing Party"), and [Insert Consultant's Name], having an address at [Insert Consultant's Address] ("Receiving Party").

**1. Definition of Confidential Information**  
For purposes of this Agreement, "Confidential Information" means any and all information disclosed by the Disclosing Party to the Receiving Party, either directly or indirectly, that is designated as confidential or that ought to be considered confidential due to the nature of the information or the circumstances surrounding its disclosure. Confidential Information includes, but is not limited to, business plans, financial data, customer lists, trade secrets, technical information, and any other proprietary information.

**2. Obligations of the Receiving Party**  
The Receiving Pa

---
## 14. Advanced: Custom Reply Functions

You can intercept the normal reply process and inject custom logic using `register_reply`.

In [ ]:
# ================================
# IMPORTS
# ================================
from datetime import datetime
from google.colab import userdata

from autogen_agentchat.agents import AssistantAgent
from autogen_ext.models.openai import OpenAIChatCompletionClient


# ================================
# MODEL CLIENT
# ================================
api_key = userdata.get("OPENAI_API_KEY")

model_client = OpenAIChatCompletionClient(
    model="gpt-4o-mini",
    api_key=api_key,
)


# ================================
# MIDDLEWARE FUNCTIONS
# ================================
def log_message(message: str, sender: str, recipient: str):
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    print(f"[LOG {timestamp}] {sender} → {recipient}")
    print(f"[LOG] Preview: {message[:100]}...\n")


def validate_input(message: str):
    disallowed = ["competitor_secret", "internal_pricing_db"]
    for keyword in disallowed:
        if keyword in message.lower():
            return False, f"Blocked due to restricted keyword: {keyword}"
    return True, None


# ================================
# AGENT
# ================================
agent = AssistantAgent(
    name="MonitoredAssistant",
    system_message="""
You are a helpful assistant.
Always answer the user's question clearly and completely.
End every response with TERMINATE.
""",
    model_client=model_client,
)


# ================================
# SAFE RUN WRAPPER
# ================================
async def safe_run(user_input: str):
    # --- validation ---
    is_valid, error = validate_input(user_input)
    if not is_valid:
        print(error)
        return

    # --- logging input ---
    log_message(user_input, "User", "Agent")

    # --- run agent directly (FIXED) ---
    result = await agent.run(task=user_input)

    final_msg = result.messages[-1].content

    # --- logging output ---
    log_message(final_msg, "Agent", "User")

    print("Response:\n")
    print(final_msg)


# ================================
# RUN
# ================================
await safe_run("What are three best practices for writing maintainable Python code?")

[LOG 2026-07-22 06:16:15] User → Agent
[LOG] Preview: What are three best practices for writing maintainable Python code?...

[LOG 2026-07-22 06:16:20] Agent → User
[LOG] Preview: Here are three best practices for writing maintainable Python code:

1. **Follow PEP 8 Guidelines**:...

Response:

Here are three best practices for writing maintainable Python code:

1. **Follow PEP 8 Guidelines**: Adhere to the Python Enhancement Proposal 8 (PEP 8) style guide. This includes consistent indentation, naming conventions for variables and functions, appropriate line lengths, and spacing. Consistent style helps readability and makes it easier for others (and yourself) to understand the code later.

2. **Write Modular and Reusable Code**: Break your code into smaller, reusable functions and classes. Each function should perform a single task or handle a specific piece of logic, which makes testing and debugging easier. Use meaningful names for functions and classes to indicate their purpose clea

In [ ]:
!pip install -U "autogen-ext[openai]"

In [ ]:
from autogen_ext.models.openai import OpenAIChatCompletionClient

In [ ]:
import autogen_ext
print(autogen_ext.__version__)

0.7.5


In [ ]:
# ============================================
# INSTALL
# ============================================
# !pip install -U pyautogen autogen-ext[openai]

# ============================================
# IMPORTS
# ============================================
from google.colab import userdata

from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.conditions import TextMentionTermination
from autogen_agentchat.teams import SelectorGroupChat

from autogen_ext.models.openai import OpenAIChatCompletionClient

# ============================================
# MODEL
# ============================================
api_key = userdata.get("OPENAI_API_KEY")

model_client = OpenAIChatCompletionClient(
    model="gpt-4o-mini",
    api_key=api_key,
)

# ============================================

In [ ]:
# AGENTS
# ============================================

researcher = AssistantAgent(
    name="Researcher",
    model_client=model_client,
    system_message="""
You are a market researcher.

Collect relevant facts and market insights.

Do NOT make recommendations.
""",
)

analyst = AssistantAgent(
    name="Analyst",
    model_client=model_client,
    system_message="""
You are a business analyst.

Use the research findings to perform SWOT analysis
and estimate risks.

Do NOT make final recommendations.
""",
)

advisor = AssistantAgent(
    name="Advisor",
    model_client=model_client,
    system_message="""
You are a senior business advisor.

Read everyone's work and provide the final recommendation.

End your response with TERMINATE.
""",
)

In [ ]:
# ============================================
# TERMINATION
# ============================================

termination = TextMentionTermination("TERMINATE")

# ============================================
# SELECTOR GROUP CHAT
# ============================================

team = SelectorGroupChat(
    participants=[
        researcher,
        analyst,
        advisor,
    ],
    model_client=model_client,
    termination_condition=termination,
    max_turns=10,
)

# ============================================
# TASK
# ============================================

task = """
A startup wants to launch an AI-powered
resume screening platform.

Research the market,
analyze strengths and risks,
and provide a final recommendation.
"""

In [ ]:
# ============================================
# RUN
# ============================================

result = await team.run(task=task)

print("\n========== FINAL CHAT ==========\n")

for msg in result.messages:
    print(f"{msg.source}:")
    print(msg.content)
    print("-" * 80)


========== FINAL CHAT ==========

user:

A startup wants to launch an AI-powered
resume screening platform.

Research the market,
analyze strengths and risks,
and provide a final recommendation.

--------------------------------------------------------------------------------
Analyst:
**SWOT Analysis for AI-Powered Resume Screening Platform**

**Strengths:**
1. **Efficiency and Speed**: The platform can process numerous resumes quickly, reducing the time HR teams spend on screening candidates.
2. **Cost-effective**: Automation can lead to cost savings in recruitment processes compared to traditional methods.
3. **Consistency**: The use of AI can minimize human biases, ensuring a more standardized evaluation of candidates.
4. **Scalability**: The platform can handle varying volumes of resumes, making it suitable for companies of all sizes.
5. **Data Insights**: AI can provide valuable insights and analytics about hiring trends and candidate qualifications.

**Weaknesses:**
1. **Initial

---
## 15. Best Practices and Common Pitfalls

### Best Practices

**Agent Design**
- Always include a termination instruction in the `system_message` (e.g., "when complete, say TERMINATE"). Without it, conversations may run indefinitely.
- Use specific, detailed `system_message` values. Generic instructions produce generic output.
- Separate execution from reasoning: use `AssistantAgent` for LLM reasoning and `UserProxyAgent` for code execution. Mixing both in one agent creates unpredictable behavior.

**Conversation Design**
- Always set `max_consecutive_auto_reply` and `max_turns` as safety guards against runaway loops.
- Use `is_termination_msg` with a specific, unambiguous trigger phrase. Avoid phrases that might appear naturally in text.
- For `GroupChat`, keep agent count low (3-5). More agents increase LLM routing confusion and cost.

**Code Execution**
- Use `use_docker=True` in production environments to sandbox code execution safely.
- Set `timeout` to prevent runaway code from blocking the conversation.
- Use `last_n_messages` to control how far back the executor searches for code blocks.

**RAG Configuration**
- Use `chunk_token_size=300` to 500 for Q&A tasks. Larger chunks work better for code retrieval.
- Set `get_or_create=True` to avoid re-indexing documents on every run.
- Always pair `RetrieveUserProxyAgent` with `RetrieveAssistantAgent` — the proxy handles retrieval, the assistant handles generation.

### Common Pitfalls

| Pitfall | Cause | Fix |
|---|---|---|
| Conversation never ends | No termination condition | Add `is_termination_msg` and TERMINATE in system prompt |
| Code blocks not executed | `code_execution_config=False` on executor | Set `code_execution_config` with `work_dir` |
| GroupChat loops on one agent | `speaker_selection_method="auto"` confused | Try `round_robin` or provide explicit selection hints in `system_message` |
| Function call not triggered | Missing `tools` in `llm_config` | Add function schemas to `llm_config["tools"]` |
| RAG retrieves wrong chunks | Chunk size too large or too small | Adjust `chunk_token_size` and `n_results` |
| High token costs in GroupChat | Too many agents, too many rounds | Reduce `max_round` and number of agents |